# Exploração · `bronze.leituras`

Notebook de **investigação**, fora do pipeline. Descobre os problemas de qualidade de `leituras` e a regra de limpeza de cada um, antes de escrever a Silver.

**Status: exploração concluída.** Rode com **Run all**: cada célula só depende das anteriores. Os números em "Achado" vêm das execuções já feitas. Se algo divergir ao rodar de novo, confira antes de usar.

| # | Seção | Resultado |
|---|---|---|
| 1 | Visão geral | Tudo é texto na Bronze |
| 2 | Nulos e texto no lugar de número | `leitura_anterior` tem nulos e `N/A`/`NULL` |
| 3 | Categorias | `tipo_leitura` sem variações |
| 4 | Formatos de data | `data_leitura` em 4 formatos |
| 5 | Duplicatas | ~21,5 mil cópias exatas |
| 6 | Integridade referencial | 8.609 linhas órfãs; cria `limpo`; ordem das datas |
| 7 | `leitura_anterior` ausente | 4 formas de ausência |
| 8 | Consumo × leituras | ±1 kWh é arredondamento |
| 9 | Consumo negativo | sinal invertido × troca de medidor |
| 10 | Troca de medidor | confirmada no cadastro |
| 11 | Validação com faturas | `leitura_atual` serve como consumo |
| 12 | Resumo | regras propostas para a Silver |

**Sobre os números:** as seções 1 a 5, 8 e 9 rodam sobre a tabela bruta (com cópias e órfãs). A partir da Seção 6 passa-se a usar `limpo` (sem cópias e sem órfãs).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"

leituras = spark.table(f"{bronze_schema}.leituras")
faturas = spark.table(f"{bronze_schema}.faturas")
medidores = spark.table(f"{bronze_schema}.medidores")

## 1. Visão geral

**O que explora:** colunas, tipos e uma amostra. Na Bronze todas as colunas são `string`, e há duas colunas de auditoria (`source_file`, `ingestion_timestamp`) que não vêm do sistema de origem.

In [0]:
leituras.printSchema()
leituras.limit(5).display()
print(f"{leituras.count():,} linhas")

root
 |-- id_leitura: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- id_medidor: string (nullable = true)
 |-- data_leitura: string (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- leitura_anterior: string (nullable = true)
 |-- leitura_atual: string (nullable = true)
 |-- consumo_kwh: string (nullable = true)
 |-- tipo_leitura: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



id_leitura,id_uc,id_medidor,data_leitura,mes_referencia,leitura_anterior,leitura_atual,consumo_kwh,tipo_leitura,source_file,ingestion_timestamp
LT000000030,UC100000,MD500000,2025-01-18,2025-01,16946,17373,426,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z
LT000000031,UC100000,MD500000,2025-02-06,2025-02,17373,17726,353,ESTIMADA,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z
LT000000032,UC100000,MD500000,2025-03-25,2025-03,17726,18132,407,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z
LT000000033,UC100000,MD500000,2025-04-22,2025-04,18132,18496,363,ESTIMADA,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z
LT000000034,UC100000,MD500000,2025-05-10,2025-05,18496,18753,257,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z


2,182,468 linhas


## 2. Nulos e texto no lugar de número

**O que explora:** células vazias por coluna e colunas numéricas que guardam texto.

**Achado:**
- `leitura_anterior` tem **8.218 nulos reais**, além dos textos `N/A` (8.269) e `NULL` (8.180).
- `leitura_atual` não tem texto inválido.
- Em `consumo_kwh`, os valores que o `bigint` recusa são números negativos escritos com `.0` (ex.: `-125.0`), e não texto. Veja a Seção 9.

In [0]:
# Nulos reais por coluna
leituras.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in leituras.columns]).display()

id_leitura,id_uc,id_medidor,data_leitura,mes_referencia,leitura_anterior,leitura_atual,consumo_kwh,tipo_leitura,source_file,ingestion_timestamp
0,0,0,0,0,8218,0,0,0,0,0


In [0]:
# Valores que não viram inteiro (texto como 'N/A' e 'NULL', ou número com casa decimal)
for c in ["leitura_anterior", "leitura_atual", "consumo_kwh"]:
    invalidos = leituras.filter(F.col(c).isNotNull() & F.expr(f"try_cast({c} as bigint)").isNull())
    print(f"\n{c}: {invalidos.count():,} valores que não viram inteiro")
    invalidos.groupBy(c).count().orderBy(F.desc("count")).limit(10).display()


leitura_anterior: 16,449 valores que não viram inteiro


leitura_anterior,count
N/A,8269
NULL,8180



leitura_atual: 0 valores que não viram inteiro


leitura_atual,count



consumo_kwh: 1,739 valores que não viram inteiro


consumo_kwh,count
-125.0,14
-65.0,13
-71.0,13
-171.0,13
-137.0,12
-106.0,12
-79.0,12
-134.0,12
-85.0,11
-113.0,11


## 3. Categorias

**O que explora:** valores distintos de `tipo_leitura`, procurando o mesmo valor escrito de formas diferentes.

**Achado:** sem variações de grafia.

In [0]:
leituras.groupBy("tipo_leitura").count().orderBy(F.desc("count")).display()

tipo_leitura,count
REAL,1786380
ESTIMADA,396088


## 4. Formatos de data

**O que explora:** troca cada dígito por `9` para mostrar os formatos de cada coluna. Mais de um padrão na mesma coluna significa datas misturadas.

**Achado (tabela bruta):**
- `data_leitura` tem **4 formatos**: ISO `yyyy-MM-dd` (1.920.533 linhas) e três formatos não ISO: `99-99-9999` (87.144), `99.99.9999` (87.618) e `99/99/9999` (87.173), cerca de 12% da tabela.
- `mes_referencia` está limpo: as 2.182.468 linhas seguem `yyyy-MM`.
- A **ordem dia/mês** dos formatos não ISO é conferida na Seção 6.5, depois que `limpo` existir.

In [0]:
(leituras.withColumn("padrao", F.regexp_replace("data_leitura", "[0-9]", "9"))
         .groupBy("padrao").count().display())

padrao,count
99-99-9999,87144
99.99.9999,87618
99/99/9999,87173
9999-99-99,1920533


In [0]:
(leituras.withColumn("padrao", F.regexp_replace("mes_referencia", "[0-9]", "9"))
         .groupBy("padrao").count().display())

padrao,count
9999-99,2182468


## 5. Duplicatas

**O que explora:** se `id_leitura` se repete e se a mesma UC aparece mais de uma vez no mesmo mês. Compara as linhas completas (sem as colunas de auditoria) para saber se as repetições são cópias exatas.

**Achado:**
- 2.182.468 linhas e 2.160.945 `id_leitura` distintos: **~21,5 mil cópias**. As linhas distintas (sem auditoria) também são 2.160.945, então as cópias são **idênticas** e `dropDuplicates(["id_leitura"])` é seguro.
- Só 2.152.336 pares `(id_uc, mes_referencia)`: **~8,6 mil linhas a mais**, com `id_leitura` diferente. É outro tipo de duplicata, explicado na Seção 6.

In [0]:
cols = [c for c in leituras.columns if c not in ("source_file", "ingestion_timestamp")]
print("linhas:", leituras.count())
print("id_leitura distintos:", leituras.select("id_leitura").distinct().count())
print("linhas distintas (todas as colunas, sem auditoria):", leituras.select(cols).distinct().count())
print("pares (id_uc, mes_referencia) distintos:", leituras.select("id_uc", "mes_referencia").distinct().count())

linhas: 2182468
id_leitura distintos: 2160945
linhas distintas (todas as colunas, sem auditoria): 2160945
pares (id_uc, mes_referencia) distintos: 2152336


In [0]:
# Exemplos de UC + mês com mais de uma linha, já sem as cópias exatas
leituras_d = leituras.dropDuplicates(["id_leitura"])

dups = leituras_d.groupBy("id_uc", "mes_referencia").count().filter("count > 1")
print(f"pares com mais de uma linha: {dups.count():,}")

(leituras_d.join(dups.select("id_uc", "mes_referencia"), ["id_uc", "mes_referencia"])
           .orderBy("id_uc", "mes_referencia")
           .limit(20).display())

pares com mais de uma linha: 8,609


id_uc,mes_referencia,id_leitura,id_medidor,data_leitura,leitura_anterior,leitura_atual,consumo_kwh,tipo_leitura,source_file,ingestion_timestamp
UC100005,2025-04,LTX0000284,MD9498450,2025-04-26,21511,21661,150,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z
UC100005,2025-04,LT000000273,MD500005,2025-04-26,21511,21661,150,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z
UC100008,2024-03,LT000000404,MD500008,2024-03-09,109187,110464,1277,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2024.csv,2026-10-19T20:11:27.087Z
UC100008,2024-03,LTX0000184,MD9396532,2024-03-09,109187,110464,1277,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2024.csv,2026-10-19T20:11:27.087Z
UC100023,2023-01,LTX0004060,MD9403537,08.01.2023,48765,49039,274,ESTIMADA,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2023.csv,2026-10-19T20:11:27.087Z
UC100023,2023-01,LT000001110,MD500023,08.01.2023,48765,49039,274,ESTIMADA,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2023.csv,2026-10-19T20:11:27.087Z
UC100024,2023-10,LT000001167,MD500024,2023-10-21,3846,3907,62,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2023.csv,2026-10-19T20:11:27.087Z
UC100024,2023-10,LTX0000677,MD9704418,2023-10-21,3846,3907,62,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2023.csv,2026-10-19T20:11:27.087Z
UC100028,2022-11,LTX0005486,MD9572561,2022-11-11,63758,63978,220,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2022.csv,2026-10-19T20:11:27.087Z
UC100028,2022-11,LT000001348,MD500028,2022-11-11,63758,63978,220,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2022.csv,2026-10-19T20:11:27.087Z


## 6. Integridade referencial

**O que explora:** se `id_medidor` e `id_uc` das leituras existem nos cadastros, e se as datas dos formatos não ISO têm a ordem esperada. O case exige que nenhuma análise dependa de registros órfãos.

### 6.1 e 6.2: formato e existência do `id_medidor`
**Achado:** o id válido é `MD` + 6 dígitos. **8.609 linhas** estão fora do formato, cada uma em par com uma linha válida da mesma UC e do mesmo mês, e **nenhum** desses ids existe em `medidores`. São órfãs e explicam o segundo tipo de duplicata da Seção 5.

In [0]:
# 6.1 Formato do id_medidor e quem aparece duplicado por UC + mês
leituras_fmt = leituras_d.withColumn("fmt_ok", F.col("id_medidor").rlike("^MD[0-9]{6}$"))
leituras_fmt.groupBy("fmt_ok").count().display()

w2 = Window.partitionBy("id_uc", "mes_referencia")
d = leituras_fmt.withColumn("n_uc_mes", F.count(F.lit(1)).over(w2)).filter("n_uc_mes > 1")
d.groupBy("n_uc_mes", "fmt_ok").count().display()

fmt_ok,count
false,8609
true,2152336


n_uc_mes,fmt_ok,count
2,false,8609
2,true,8609


In [0]:
# 6.2 O id_medidor existe no cadastro? (dropDuplicates para não multiplicar linhas: medidores tem ids repetidos)
med = medidores.select("id_medidor").distinct().withColumn("existe", F.lit(True))
(leituras_fmt.join(med, "id_medidor", "left")
   .groupBy("fmt_ok", F.coalesce("existe", F.lit(False)).alias("existe_em_medidores"))
   .count().display())

fmt_ok,existe_em_medidores,count
false,false,8609
true,true,2152336


### 6.3 Tabela `limpo`
Sem cópias e sem órfãs. **A partir daqui as análises usam `limpo`.** Esperado: 2.152.336 linhas e 2.152.336 pares `(id_uc, mes_referencia)`, ou seja, a chave fica única.

In [0]:
limpo = leituras_fmt.filter("fmt_ok")
print("linhas:", limpo.count(), "| pares (uc, mês) distintos:", limpo.select("id_uc", "mes_referencia").distinct().count())

linhas: 2152336 | pares (uc, mês) distintos: 2152336


### 6.4 O `id_uc` existe no cadastro?
**Achado:** todas as 2.152.336 linhas de `limpo` têm `id_uc` em `unidades_consumidoras`. Não há órfãs de UC, só as de medidor.

In [0]:
ucs = (spark.table(f"{bronze_schema}.unidades_consumidoras")
            .select("id_uc").distinct().withColumn("existe_uc", F.lit(True)))

(limpo.join(ucs, "id_uc", "left")
      .groupBy(F.coalesce("existe_uc", F.lit(False)).alias("uc_existe_no_cadastro"))
      .count().display())

uc_existe_no_cadastro,count
true,2152336


### 6.5 Ordem dia/mês dos formatos de data
Converte cada padrão com o seu formato e compara o mês da data com o `mes_referencia`. Se a ordem estiver certa, o mês bate em praticamente todas as linhas.

**Achado:** os quatro padrões convertem sem falhas e o mês bate em 100% das linhas:
- `yyyy-MM-dd` (ISO): 1.894.056
- `dd.MM.yyyy`: 86.433
- `dd/MM/yyyy`: 85.936
- `MM-dd-yyyy` (**mês primeiro**): 85.911

Com `dd-MM-yyyy` no padrão com hífen, 49.082 das 85.911 linhas não convertiam e só 3.132 batiam com o mês de referência. Na Silver cada padrão precisa do seu formato: um formato único converteria datas para o mês errado, sem dar erro.

In [0]:
dt = limpo.withColumn("padrao", F.regexp_replace("data_leitura", "[0-9]", "9"))

dt = dt.withColumn("data_parse",
    F.when(F.col("padrao") == "9999-99-99", F.expr("try_to_timestamp(data_leitura, 'yyyy-MM-dd')"))
     .when(F.col("padrao") == "99-99-9999", F.expr("try_to_timestamp(data_leitura, 'MM-dd-yyyy')"))
     .when(F.col("padrao") == "99.99.9999", F.expr("try_to_timestamp(data_leitura, 'dd.MM.yyyy')"))
     .when(F.col("padrao") == "99/99/9999", F.expr("try_to_timestamp(data_leitura, 'dd/MM/yyyy')")))

(dt.withColumn("mes_da_data", F.date_format("data_parse", "yyyy-MM"))
   .groupBy("padrao")
   .agg(F.count("*").alias("linhas"),
        F.sum(F.col("data_parse").isNull().cast("int")).alias("nao_converteu"),
        F.sum((F.col("mes_da_data") == F.col("mes_referencia")).cast("int")).alias("mes_bate_com_referencia"))
   .display())

padrao,linhas,nao_converteu,mes_bate_com_referencia
99-99-9999,85911,0,85911
99.99.9999,86433,0,86433
99/99/9999,85936,0,85936
9999-99-99,1894056,0,1894056


## 7. `leitura_anterior` ausente

**O que explora:** de quantas formas a leitura anterior aparece "vazia", se o consumo continua preenchido e se os casos são a primeira leitura da UC (ausência legítima).

**Achado (em `limpo`):**
- **Quatro formas de ausência:** nulo real (8.101), texto `N/A` (8.149), texto `NULL` (8.073) e o código `-1` (7.962). São 32.285 linhas, 1,5% da tabela, e o consumo está preenchido e numérico em todas.
- Dos `N/A`/`NULL` (16.222), só **339 são a primeira leitura da UC**. É dado faltando, e não ausência de negócio.
- Distribuição uniforme: ~340 por mês (~2 mil nos arquivos de 2022 e 2026, que têm 6 meses; ~4 mil nos anos completos). Por tipo: 13.248 REAL e 2.974 ESTIMADA.
- As 8.042 linhas em que o consumo não bate com as leituras (Seção 8) têm todas `leitura_anterior = -1`, e o consumo é coerente.

**Decisão proposta:** converter as quatro formas em nulo real, criar a flag `leitura_anterior_ausente` e manter o consumo.

In [0]:
# As quatro formas de ausência, e se o consumo está preenchido em cada uma
forma = (F.when(F.col("leitura_anterior").isNull(), "nulo_real")
          .when(F.col("leitura_anterior") == "N/A", "texto_N/A")
          .when(F.col("leitura_anterior") == "NULL", "texto_NULL")
          .when(F.col("leitura_anterior") == "-1", "codigo_-1")
          .otherwise("preenchida"))

(limpo.withColumn("forma", forma)
      .withColumn("consumo_numerico", F.expr("try_cast(consumo_kwh as double)").isNotNull())
      .groupBy("forma", "consumo_numerico").count().orderBy("forma").display())

forma,consumo_numerico,count
codigo_-1,true,7962
nulo_real,true,8101
preenchida,true,2120051
texto_N/A,true,8149
texto_NULL,true,8073


In [0]:
# Os 'N/A' e 'NULL' são a primeira leitura da UC? Concentram-se em algum tipo ou arquivo?
sem_ant = limpo.filter(F.col("leitura_anterior").isin("N/A", "NULL"))
primeira = limpo.groupBy("id_uc").agg(F.min("mes_referencia").alias("primeiro_mes"))

print("N/A ou NULL:", sem_ant.count(),
      "| são a 1ª leitura da UC:", sem_ant.join(primeira, "id_uc").filter("mes_referencia = primeiro_mes").count())

sem_ant.groupBy("tipo_leitura").count().display()
sem_ant.groupBy("source_file").count().display()

N/A ou NULL: 16222 | são a 1ª leitura da UC: 339


tipo_leitura,count
ESTIMADA,2974
REAL,13248


source_file,count
dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2026.csv,2054
dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2024.csv,4141
dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,3944
dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2023.csv,4073
dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2022.csv,2010


## 8. Consumo × leituras

**O que explora:** se `consumo_kwh = leitura_atual − leitura_anterior`. Usa `try_cast`, porque o `cast` comum dá erro com `N/A`/`NULL`.

**Achado:**
- Entre os consumos não negativos, **1.616.562 batem exatamente**, 262.570 têm diferença de +1 e 262.566 de −1. O ±1 é simétrico (~24% das linhas): parece **arredondamento da origem**. Não recalcular o consumo; a validação deve tolerar ±1.
- A "cauda" (|diferença| > 1) tem 8.042 linhas, e **todas** têm `leitura_anterior = -1`.

In [0]:
leituras_num = (leituras
    .withColumn("consumo", F.expr("try_cast(consumo_kwh as double)"))
    .withColumn("atual", F.expr("try_cast(leitura_atual as bigint)"))
    .withColumn("anterior", F.expr("try_cast(leitura_anterior as bigint)")))

# Consumos não negativos que podem ser conferidos com as leituras
ok = (leituras_num
    .filter("consumo >= 0")
    .withColumn("calc", F.col("atual") - F.col("anterior"))
    .filter("calc IS NOT NULL")
    .withColumn("dif", F.col("consumo") - F.col("calc")))

ok.groupBy("dif").count().orderBy(F.desc("count")).limit(10).display()

dif,count
0.0,1616562
1.0,262570
-1.0,262566
-71669.0,4
-62056.0,3
-90828.0,3
-71806.0,3
-40531.0,3
-84384.0,3
-22051.0,3


In [0]:
# A cauda (|dif| > 1): o que explica?
cauda = ok.filter(F.abs("dif") > 1)
print("cauda:", cauda.count(),
      "| com leitura_anterior = -1:", cauda.filter(F.col("leitura_anterior") == "-1").count(),
      "| total de -1 na tabela:", leituras.filter(F.col("leitura_anterior") == "-1").count())

cauda.withColumn("ano", F.substring("mes_referencia", 1, 4)).groupBy("ano").count().orderBy("ano").display()
cauda.select("id_uc", "id_medidor", "mes_referencia", "leitura_anterior", "leitura_atual", "consumo_kwh", "tipo_leitura").limit(10).display()

cauda: 8042 | com leitura_anterior = -1: 8042 | total de -1 na tabela: 8068


ano,count
2022,1002
2023,2036
2024,2027
2025,1991
2026,986


id_uc,id_medidor,mes_referencia,leitura_anterior,leitura_atual,consumo_kwh,tipo_leitura
UC100017,MD500017,2025-09,-1,37206,110,REAL
UC100063,MD500063,2025-04,-1,52268,219,REAL
UC100099,MD500099,2025-12,-1,32572,258,REAL
UC100148,MD500148,2025-06,-1,91657,80,REAL
UC100174,MD500174,2025-09,-1,22794,81,REAL
UC100275,MD500275,2025-05,-1,20454,118,REAL
UC100286,MD500286,2025-02,-1,75759,500,REAL
UC100299,MD500299,2025-02,-1,60398,196,REAL
UC100310,MD500310,2025-08,-1,37860,156,REAL
UC100354,MD500354,2025-01,-1,19110,259,REAL


## 9. Consumo negativo

**O que explora:** por que há consumo negativo, que não faz sentido físico. Classifica cada linha negativa (com tolerância de ±1) em `sinal_invertido` (leituras coerentes, consumo com sinal trocado), `leitura_regrediu` (leitura atual menor que a anterior), `anterior_nulo` e `outro`.

**Achado:** **8.150** consumos negativos (zero consumos iguais a zero).
- **`sinal_invertido`: 1.708 linhas, todas escritas com `.0`** (ex.: `-85.0`). Com as 22 de `anterior_nulo` e as 9 de `outro`, são 1.739: todo valor com `.0` é negativo, e o `.0` marca o defeito de sinal.
- **`leitura_regrediu`: 6.327 linhas, nenhuma com `.0`.** A leitura anterior é alta (mediana ~55 mil) e a atual muito baixa. Não é virada de mostrador (que ocorreria perto de um teto como 99999). Aparece **só de 2025-09 a 2026-03**, ~900 por mês: é troca de medidor (Seção 10).
- A tolerância de ±1 reduz o grupo `outro` de 1.946 para 26.

**Importante na Silver:** o critério do `.0` depende do texto original. Classifique **antes** do cast para número.

In [0]:
print("negativos:", leituras_num.filter("consumo < 0").count())
print("zeros:", leituras_num.filter("consumo = 0").count())
print("consumo preenchido que não virou número:",
      leituras_num.filter(F.col("consumo_kwh").isNotNull() & F.col("consumo").isNull()).count())

negativos: 8150
zeros: 0
consumo preenchido que não virou número: 0


In [0]:
# Classificação dos consumos negativos (tolerância de ±1) e marca ".0"
base = (leituras_num
    .filter("consumo < 0")
    .withColumn("calc", F.col("atual") - F.col("anterior"))
    .withColumn("termina_em_ponto_zero", F.col("consumo_kwh").endswith(".0")))

base = base.withColumn("categoria",
    F.when(F.col("anterior").isNull(), "anterior_nulo")
     .when(F.abs(F.col("consumo") - F.col("calc")) <= 1, "leitura_regrediu")
     .when(F.abs(F.col("consumo") + F.col("calc")) <= 1, "sinal_invertido")
     .otherwise("outro"))

base.groupBy("categoria", "termina_em_ponto_zero").count().orderBy("categoria").display()

categoria,termina_em_ponto_zero,count
anterior_nulo,false,67
anterior_nulo,true,22
leitura_regrediu,false,6327
outro,true,9
outro,false,17
sinal_invertido,true,1708


In [0]:
# Amostra de cada categoria
for cat in ["leitura_regrediu", "sinal_invertido", "outro"]:
    print(cat)
    base.filter(F.col("categoria") == cat).select(
        "id_uc", "id_medidor", "mes_referencia", "leitura_anterior",
        "leitura_atual", "consumo_kwh", "tipo_leitura").limit(8).display()

leitura_regrediu


id_uc,id_medidor,mes_referencia,leitura_anterior,leitura_atual,consumo_kwh,tipo_leitura
UC100011,MD545218,2025-12,195613,4698,-190915,REAL
UC100015,MD545821,2025-10,79980,80,-79900,REAL
UC100023,MD549795,2025-12,55647,250,-55397,REAL
UC100038,MD547985,2025-11,55487,92,-55395,REAL
UC100083,MD545466,2025-12,33098,171,-32927,REAL
UC100084,MD549408,2025-11,47956,196,-47760,REAL
UC100094,MD550781,2025-10,82879,165,-82714,REAL
UC100119,MD549062,2025-09,50422,68,-50354,REAL


sinal_invertido


id_uc,id_medidor,mes_referencia,leitura_anterior,leitura_atual,consumo_kwh,tipo_leitura
UC100005,MD500005,2025-01,20911,21100,-190.0,REAL
UC100121,MD500121,2025-10,15007,15092,-85.0,ESTIMADA
UC100177,MD500177,2025-08,56566,57421,-854.0,ESTIMADA
UC100241,MD500241,2025-05,58449,58771,-322.0,REAL
UC100268,MD500268,2025-12,54015,54197,-182.0,REAL
UC100292,MD500292,2025-03,59735,60203,-469.0,REAL
UC100605,MD500605,2025-02,94076,94275,-199.0,REAL
UC100626,MD500626,2025-07,72877,72957,-81.0,REAL


outro


id_uc,id_medidor,mes_referencia,leitura_anterior,leitura_atual,consumo_kwh,tipo_leitura
UC103380,MD545724,2025-10,-1,71,-27917,REAL
UC118139,MD549725,2025-10,-1,183,-46243,REAL
UC123333,MD545609,2025-09,-1,60,-36000,REAL
UC126386,MD547476,2025-10,-1,121,-41619,REAL
UC128583,MD547968,2025-11,-1,227,-85028,REAL
UC129861,MD547445,2025-10,-1,84,-50917,REAL
UC132545,MD545027,2025-12,-1,92,-31993,REAL
UC133874,MD547152,2025-09,-1,168,-16274,REAL


In [0]:
# Virada de mostrador? Se fosse, a leitura anterior ficaria perto de um teto (ex.: 99999)
(base.filter("categoria = 'leitura_regrediu'")
     .agg(F.min("anterior"), F.expr("percentile_approx(anterior, 0.5)"), F.max("anterior"),
          F.min("atual"), F.max("atual")).display())

min(anterior),"percentile_approx(anterior, 0.5, 10000)",max(anterior),min(atual),max(atual)
3593,56018,2156270,28,58011


In [0]:
# Quando cada categoria acontece: por ano e, para a regressão de leitura, por mês
(base.withColumn("ano", F.substring("mes_referencia", 1, 4))
     .groupBy("categoria", "ano").count().orderBy("categoria", "ano").display())

(base.filter("categoria = 'leitura_regrediu'")
     .groupBy("mes_referencia").count().orderBy("mes_referencia").display())

categoria,ano,count
anterior_nulo,2022,4
anterior_nulo,2023,3
anterior_nulo,2024,4
anterior_nulo,2025,48
anterior_nulo,2026,30
leitura_regrediu,2025,3612
leitura_regrediu,2026,2715
outro,2022,2
outro,2023,5
outro,2025,9


mes_referencia,count
2025-09,923
2025-10,886
2025-11,907
2025-12,896
2026-01,916
2026-02,875
2026-03,924


## 10. Troca de medidor

**O que explora:** se o `id_medidor` de uma UC muda de um mês para o outro, se isso coincide com a regressão de leitura (Seção 9) e se o cadastro de `medidores` confirma a troca. `leitura_continua = true` significa que a leitura anterior da linha é igual à leitura atual da linha anterior.

### 10.1 a 10.3: trocas em `limpo`
**Achado:**
- **6.299 trocas**, todas em 2025 (3.602) e 2026 (2.697), nenhuma antes. Em 6.215 delas (~98,7%) a leitura anterior é a última do medidor antigo e a atual é a do novo, por isso `atual − anterior` não faz sentido.
- Antes de remover as órfãs havia 23.314 mudanças de medidor. A maior parte era causada pelas linhas órfãs, que faziam o id "pular" para um id inválido e voltar.
- **Separação total (10.2):** das linhas `leitura_regrediu`, 6.215 de 6.215 tiveram troca de medidor, e das `sinal_invertido`, 0 de 1.687. As 6.299 trocas estão todas entre os consumos negativos (6.215 + 67 + 17). Os 1.717 negativos sem troca (1.687 + 22 + 8) são sinal invertido. O critério do medidor e o do `.0` dão a mesma classificação.

In [0]:
# 10.1 Trocas de medidor em `limpo` (sem cópias e sem órfãs)
w = Window.partitionBy("id_uc").orderBy("mes_referencia")

trocas = (limpo
    .withColumn("medidor_ant", F.lag("id_medidor").over(w))
    .withColumn("atual_ant", F.lag("leitura_atual").over(w))
    .withColumn("mudou_medidor", F.col("medidor_ant").isNotNull() & (F.col("id_medidor") != F.col("medidor_ant")))
    .withColumn("leitura_continua", F.col("leitura_anterior") == F.col("atual_ant")))

t = trocas.filter("mudou_medidor").withColumn("ano", F.substring("mes_referencia", 1, 4))
print("trocas de medidor em limpo:", t.count())
t.groupBy("ano", "leitura_continua").count().orderBy("ano").display()

trocas de medidor em limpo: 6299


ano,leitura_continua,count
2025,false,37
2025,null,12
2025,true,3553
2026,true,2662
2026,false,23
2026,null,12


In [0]:
# 10.2 A troca de medidor coincide com a regressão de leitura?
(base.dropDuplicates(["id_leitura"]).select("id_leitura", "categoria")
     .join(trocas.select("id_leitura", "mudou_medidor"), "id_leitura")
     .groupBy("categoria", "mudou_medidor").count().orderBy("categoria").display())

categoria,mudou_medidor,count
anterior_nulo,false,22
anterior_nulo,true,67
leitura_regrediu,true,6215
outro,true,17
outro,false,8
sinal_invertido,false,1687


In [0]:
# 10.3 Evidência das órfãs: linha do tempo de medidores de 3 UCs que têm id inválido
ex = leituras_d.filter(~F.col("id_medidor").rlike("^MD[0-9]{6}$")).select("id_uc").distinct().limit(3)

(leituras_d.join(ex, "id_uc")
   .groupBy("id_uc", "id_medidor")
   .agg(F.min("mes_referencia").alias("de"), F.max("mes_referencia").alias("ate"), F.count("*").alias("n"))
   .orderBy("id_uc", "de").display())

id_uc,id_medidor,de,ate,n
UC107107,MD507107,2022-07,2026-06,48
UC107107,MD9487370,2025-06,2025-06,1
UC107107,MD9963829,2025-08,2025-08,1
UC110534,MD510534,2022-07,2026-06,48
UC110534,MD9185506,2025-06,2025-06,1
UC127299,MD527299,2022-07,2026-06,48
UC127299,MD9843555,2025-10,2025-10,1


### 10.4 Cruzamento com o cadastro de `medidores`
Para cada troca de `limpo`, compara o medidor antigo e o novo com o cadastro.

**Achado** (rodado antes de deduplicar `medidores`, por isso com 6.436 linhas em vez de 6.299):
- O medidor antigo tem `status = RETIRADO` em **todas**, com `data_retirada` no **mesmo mês** da troca (0 meses de diferença em 100%).
- O medidor novo pertence à **mesma UC** no cadastro em 100%.
- A `data_instalacao` do novo cai no mesmo mês da troca em 5.970 linhas. Em 466 não foi lida, porque está em formato não ISO (ver 10.5).

Isso confirma que a regressão de leitura é **troca de medidor registrada no cadastro**, e não erro de leitura.

**Nesta versão** o cruzamento deduplica `medidores` antes do join, para não multiplicar linhas. Os totais devem dar 6.299, e as proporções devem se manter.

In [0]:
# Deduplica medidores por id (a tabela tem 513 linhas repetidas, ver 10.5) para o join não multiplicar linhas
medidores_u = medidores.dropDuplicates(["id_medidor"])

tr = (trocas.filter("mudou_medidor")
        .select("id_uc", "mes_referencia",
                F.col("medidor_ant").alias("id_medidor_antigo"),
                F.col("id_medidor").alias("id_medidor_novo"))
        .withColumn("mes_ref", F.expr("to_date(concat(mes_referencia, '-01'))")))

antigo = (medidores_u
    .withColumn("retirada", F.expr("try_to_timestamp(data_retirada, 'yyyy-MM-dd')"))
    .select(F.col("id_medidor").alias("id_medidor_antigo"), F.col("status").alias("status_antigo"), "retirada"))

novo = (medidores_u
    .withColumn("instalacao", F.expr("try_to_timestamp(data_instalacao, 'yyyy-MM-dd')"))
    .select(F.col("id_medidor").alias("id_medidor_novo"), F.col("id_uc").alias("uc_do_novo"), "instalacao"))

x = tr.join(antigo, "id_medidor_antigo", "left").join(novo, "id_medidor_novo", "left")
print("trocas:", tr.count(), "| depois dos joins:", x.count(), "(devem ser iguais)")

# 1. Status e data de retirada do medidor antigo
x.groupBy("status_antigo").count().display()
(x.withColumn("retirada_menos_ref_meses", F.round(F.months_between(F.trunc("retirada", "month"), F.col("mes_ref"))))
  .groupBy("retirada_menos_ref_meses").count().orderBy("retirada_menos_ref_meses").display())

# 2. Data de instalação do medidor novo (null = formato não ISO, ver 10.5)
(x.withColumn("instalacao_menos_ref_meses", F.round(F.months_between(F.trunc("instalacao", "month"), F.col("mes_ref"))))
  .groupBy("instalacao_menos_ref_meses").count().orderBy("instalacao_menos_ref_meses").display())

# 3. O medidor novo pertence à mesma UC no cadastro?
print("medidor novo é da mesma UC:", x.filter("uc_do_novo = id_uc").count(), "de", x.count())

trocas: 6299 | depois dos joins: 6299 (devem ser iguais)


status_antigo,count
RETIRADO,6299


retirada_menos_ref_meses,count
0.0,6299


instalacao_menos_ref_meses,count
null,457
0.0,5842


medidor novo é da mesma UC: 6299 de 6299


### 10.5 Problemas em `medidores` (para o bloco A)
Estes dois achados são do cadastro e não de `leituras`, e devem ser levados à colega do bloco A.

**Achado:**
- `medidores` tem **51.812 linhas para 51.299 ids distintos** (513 repetidas). Um join sem deduplicar multiplica linhas: foi isso que inflou o cruzamento de 6.299 para 6.436. A célula abaixo mostra se as repetidas são cópias exatas.
- `data_instalacao` está em **formatos mistos**. Entre os 6.367 medidores novos distintos, 5.907 estão em ISO e 460 (7%) em `99.99.9999` (142), `99/99/9999` (175) e `99-99-9999` (143). A ordem dia/mês desses formatos em `medidores` **não foi conferida**. Em `leituras` o padrão com hífen era mês primeiro, mas isso não pode ser assumido aqui.

In [0]:
# Medidores repetidos: cópias exatas ou ids com dados diferentes?
cols_m = [c for c in medidores.columns if c not in ("source_file", "ingestion_timestamp")]
print("linhas:", medidores.count())
print("ids distintos:", medidores.select("id_medidor").distinct().count())
print("linhas distintas (sem auditoria):", medidores.select(cols_m).distinct().count(), "(= ids distintos significa cópias exatas)")

linhas: 51812
ids distintos: 51299
linhas distintas (sem auditoria): 51299 (= ids distintos significa cópias exatas)


In [0]:
# Formatos de data_instalacao dos medidores novos
novos = (tr.select("id_medidor_novo").distinct()
           .join(medidores_u.select(F.col("id_medidor").alias("id_medidor_novo"), "data_instalacao"), "id_medidor_novo"))

(novos.withColumn("padrao", F.regexp_replace(F.coalesce("data_instalacao", F.lit("<nulo>")), "[0-9]", "9"))
      .groupBy("padrao").count().display())

padrao,count
9999-99-99,5842
99.99.9999,142
99/99/9999,173
99-99-9999,142


## 11. Validação cruzada com faturas

**O que explora:** nos meses de troca de medidor o consumo registrado é negativo, mas a fatura foi emitida normalmente. A pergunta: **a `leitura_atual` do medidor novo funciona como consumo do mês?** Dois testes:
1. `leitura_atual` ÷ consumo típico (mediana) da própria UC. Perto de 1 sugere que a leitura se parece com um consumo.
2. Preço implícito da fatura (`valor_energia ÷ leitura_atual`) ÷ preço típico da UC (`valor_energia ÷ consumo` nos meses normais). Perto de 1 sugere que a fatura foi calculada com a `leitura_atual` como consumo.

**Achado:**
- **Teste 1** (6.215 linhas): quantis de ~0,64 (10%), 0,88 (25%), **1,20 (mediana)**, 1,34 (75%) e 1,45 (90%). É compatível com um consumo mensal, mas a faixa é ampla e a mediana é 20% acima do típico. **Evidência fraca.**
- **Teste 2** (6.215 linhas): quantis de ~1,03 (10%), 1,03 (25%), **1,05 (mediana)**, 1,10 (75%) e 1,15 (90%). Faixa estreita, e o desvio de ~5% é coerente com o reajuste de tarifa dos meses recentes. **Evidência forte.**

**Conclusão provisória:** a fatura desses meses foi calculada com a `leitura_atual` como consumo. Junto com o cadastro (Seção 10.4), sustenta `consumo = leitura_atual` com flag `troca_medidor`. Não é prova: a decisão é do grupo.

In [0]:
limpo_num = (limpo
    .withColumn("consumo", F.expr("try_cast(consumo_kwh as double)"))
    .withColumn("atual", F.expr("try_cast(leitura_atual as bigint)"))
    .withColumn("anterior", F.expr("try_cast(leitura_anterior as bigint)")))

# Teste 1: leitura_atual / consumo típico da UC (nas linhas de regressão de leitura)
tipico = (limpo_num.filter("consumo > 0").groupBy("id_uc")
             .agg(F.expr("percentile_approx(consumo, 0.5)").alias("consumo_tipico")))

campanha = (limpo_num.filter("consumo < 0 AND atual < anterior").join(tipico, "id_uc")
              .withColumn("razao", F.col("atual") / F.col("consumo_tipico")))

campanha.select(F.count("*").alias("linhas"),
    F.expr("percentile_approx(razao, array(0.1, 0.25, 0.5, 0.75, 0.9))").alias("quantis_razao")).display()

linhas,quantis_razao
6215,"List(0.6412213740458015, 0.8837209302325582, 1.2011834319526626, 1.3417721518987342, 1.4538461538461538)"


In [0]:
# Teste 2: preço implícito da fatura usando leitura_atual como consumo
fat = (faturas.dropDuplicates(["id_fatura"])
         .withColumn("energia", F.expr("try_cast(valor_energia as double)"))
         .select("id_uc", "mes_referencia", "energia"))

L = limpo_num.join(fat, ["id_uc", "mes_referencia"])

preco_normal = (L.filter("consumo > 0 AND anterior IS NOT NULL AND atual >= anterior")
                 .withColumn("preco", F.col("energia") / F.col("consumo"))
                 .groupBy("id_uc").agg(F.expr("percentile_approx(preco, 0.5)").alias("preco_tipico")))

camp = (L.filter("consumo < 0 AND atual < anterior").join(preco_normal, "id_uc")
          .withColumn("razao_preco", (F.col("energia") / F.col("atual")) / F.col("preco_tipico")))

camp.select(F.count("*").alias("linhas"),
            F.expr("percentile_approx(razao_preco, array(0.1, 0.25, 0.5, 0.75, 0.9))").alias("quantis")).display()

linhas,quantis
6215,"List(1.0273972602739727, 1.0293996821976472, 1.0524750841848862, 1.0966317077073473, 1.153134815069081)"


## 12. Resumo e regras propostas para a Silver

| Problema | Linhas | Regra proposta | 
|---|---|---|
| Cópias exatas | ~21,5 mil | `dropDuplicates(["id_leitura"])` | 
| Linha órfã (id de medidor inexistente) | 8.609 | remover ou mandar para quarentena | 
| `leitura_anterior` ausente (nulo real, `N/A`, `NULL`, `-1`) | 32.285 em `limpo` | nulo real + flag `leitura_anterior_ausente`; manter o consumo | 
| Consumo negativo e o medidor **não** mudou (sinal invertido) | ~1,7 mil | `abs(consumo)` + flag `consumo_corrigido` | 
| Consumo negativo e o medidor **mudou** (troca de medidor, 2025-09 a 2026-03) | 6.299 | `consumo = leitura_atual` + flag `troca_medidor`. Apoiado no cadastro (medidor antigo `RETIRADO` no mesmo mês, 100%) e nas faturas (razão de preço ~1,05) | 
| Diferença de ±1 entre consumo e leituras | ~525 mil | não corrigir; usar como tolerância de validação |
| `data_leitura` em 4 formatos (3 não ISO) | ~258 mil | converter cada padrão com o seu formato (`yyyy-MM-dd`, `dd.MM.yyyy`, `dd/MM/yyyy`, `MM-dd-yyyy`) e gravar como `date` | 

### Ordem das operações na Silver (a ordem muda o resultado)
1. Ler o texto original de `consumo_kwh` **antes** do cast (o `.0` só existe no texto).
2. Remover cópias, depois remover órfãs (com `left_anti` contra os ids de medidor, que não multiplica linhas), e **só então** calcular a troca de medidor. Calcular antes cria trocas falsas.
3. Usar `try_cast` nas conversões.

### Para levar ao bloco A
- `medidores` tem 513 linhas com ids repetidos (confirmar se são cópias exatas, célula 10.5).
- `data_instalacao` em `medidores` está em formatos mistos, e a ordem dia/mês não foi conferida.
- O medidor antigo das trocas tem `data_retirada` correta. O `9999-12-31` que a colega encontrou é outro assunto (medidores ativos sem data de retirada).

**A exploração de `leituras` está concluída.** Novos achados devem ir para `docs/data_quality_log.md`, e não para este notebook.